# MLP Knowledge Editing

Locate and modify factual knowledge stored in MLP layers. Implements
rank-one editing (ROME-style), fact localization, and side-effect analysis.

References:
- Meng et al. (2022) "Locating and Editing Factual Associations in GPT"
- Meng et al. (2023) "Mass-Editing Memory in a Transformer"

## Why This Matters

MLP knowledge editing provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_knowledge_editing import (
    locate_fact_in_mlps,
    rank_one_mlp_edit,
    verify_edit_effect,
    edit_side_effects,
    mlp_fact_extraction,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Locate Facts in MLPs

Identify which MLP layers store a given factual association by measuring
the effect of ablating each MLP on a metric.

In [ ]:
result = locate_fact_in_mlps(model, tokens, metric_fn)
print(f"MLP effects: {result['mlp_effects']}")
print(f"Decisive layer: {result['decisive_layer']}")
print(f"Fact distributed: {result['fact_distributed']}")
print(f"\nTop layers:")
for layer, effect in result['top_layers']:
    print(f"  Layer {layer}: {effect:.4f}")

## Rank-One MLP Edit

Apply a rank-one update to MLP weights to change a factual association.
This is the core operation of ROME.

In [ ]:
d_mlp = cfg.d_model * 4
key_vec = np.random.RandomState(42).randn(d_mlp)
val_vec = np.random.RandomState(43).randn(cfg.d_model)
result = rank_one_mlp_edit(model, layer=1, key_vector=key_vec, value_vector=val_vec, scale=2.0)
print(f"Edit norm: {result['edit_norm']:.4f}")
print(f"Key norm: {result['key_norm']:.4f}")
orig_out = np.array(model(tokens))[-1, :5]
edit_out = np.array(result['edited_model'](tokens))[-1, :5]
print(f"\nOriginal logits (first 5): {orig_out}")
print(f"Edited logits (first 5): {edit_out}")

## Verify Edit Effect

Check that the edit achieves the desired metric change on target inputs.

In [ ]:
tokens_list = [tokens, jnp.array([1, 2, 3, 4, 5, 6, 7])]
verify = verify_edit_effect(model, result['edited_model'], tokens_list, metric_fn)
print(f"Original metrics: {verify['original_metrics']}")
print(f"Edited metrics: {verify['edited_metrics']}")
print(f"Changes: {verify['metric_changes']}")
print(f"Success rate: {verify['success_rate']:.2%}")

## MLP Fact Extraction

See what factual information an MLP layer contributes by projecting
its output through the unembedding.

In [ ]:
for layer in range(cfg.n_layers):
    result = mlp_fact_extraction(model, tokens, layer=layer, top_k=3)
    print(f"\nLayer {layer}: norm={result['mlp_output_norm']:.4f}, entropy={result['output_entropy']:.4f}")
    print(f"  Promoted: {result['promoted_tokens']}")
    print(f"  Suppressed: {result['suppressed_tokens']}")